# NLP-Based Smart FAQ Retrieval and Question Answering System

**CSE 4122 — Natural Language Processing Laboratory, Phase 1**

This notebook walks through the whole system end to end: how the FAQ corpora are
loaded, how text is preprocessed, how the TF-IDF index is built, how a query is
matched by cosine similarity, how the rejection threshold is chosen, and how the
final results were measured.

The system **retrieves an existing answer**. It does not generate answers.

The same engine runs over two independent corpora — University and E-commerce —
each with its own preprocessing configuration and its own threshold, both chosen
from validation data only.

## 1. Setup

Everything below reads local CSV files. No internet connection is used.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Run from the project root so `src` is importable.
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.data_loader import (
    discover_corpora,
    load_corpus_config,
    load_faq_dataset,
    load_query_dataset,
)
from src.preprocessing import preprocess_text
from src.tfidf_retrieval import answer_query, build_tfidf_index, retrieve_tfidf
from src.evaluation import PREPROCESSING_CONFIGS, evaluate_tfidf, tune_threshold

pd.set_option("display.max_colwidth", 90)
print("Project root:", ROOT)


Project root: D:\4.1\NLP\lab\Smart_FAQ


## 2. Discovering and loading the corpora

`discover_corpora` finds any directory under `data/` that contains a
`faq_dataset.csv`. Adding a third corpus needs no code change — this is what
"corpus-configurable" means in practice.

`load_faq_dataset` validates the schema on the way in: integer ids, no duplicate
ids, no blank fields, and no duplicate questions (exact or normalized).

In [2]:
corpora = discover_corpora(ROOT / "data")
print("Discovered corpora:", list(corpora))

datasets = {}
for name, directory in corpora.items():
    faq = load_faq_dataset(directory)
    config = load_corpus_config(directory)
    datasets[name] = {"faq": faq, "config": config, "dir": directory}
    print(f"\n{config['display_name']}: {len(faq)} FAQs, "
          f"{faq['category'].nunique()} categories")
    print(f"  frozen preprocessing : {config.get('preprocessing_config')}")
    print(f"  frozen threshold     : {config['similarity_threshold']:.2f}")

Discovered corpora: ['ecommerce', 'university']

E-commerce FAQ: 500 FAQs, 10 categories
  frozen preprocessing : basic
  frozen threshold     : 0.58

University FAQ: 500 FAQs, 14 categories
  frozen preprocessing : basic
  frozen threshold     : 0.46


In [3]:
datasets["university"]["faq"].head(3)

,id,question,answer,category,source,source_type
0,1,what courses are offered in Master of Visual Studies in Studio Art at UOFT?,Small class sizes and individual mentorship create a unique atmosphere where our stude...,courses_programs,https://www.daniels.utoronto.ca/programs/graduate/master-visual-studies-studio-art,public_dataset
1,2,what courses are offered in Master of Visual Studies in Curatorial Studies at UOFT?,Small class sizes and individual mentorship create a unique atmosphere where our stude...,courses_programs,https://www.daniels.utoronto.ca/programs/graduate/master-visual-studies-curatorial-stu...,public_dataset
2,3,"describe the PhD in Architecture, Landscape, and Design curriculum at UOFT.",Through our highly adaptable curriculum-one that is unlike other PhD programs in archi...,courses_programs,https://www.daniels.utoronto.ca/programs/phd/phd-architecture-landscape-and-design,public_dataset


### Where the records come from

Each record keeps the URL it came from. The university corpus deliberately
combines two source types, because the primary dataset alone could not supply
500 records that survived review. See `docs/DATA_SOURCES.md` for the full
review record.

In [4]:
for name, bundle in datasets.items():
    print(name)
    print(bundle["faq"]["source_type"].value_counts().to_string(), "\n")

ecommerce
source_type
official_web_via_public_dataset    500 

university
source_type
official_public_faq    261
public_dataset         239 



## 3. Preprocessing

`preprocess_text` lowercases, removes URLs and HTML, strips punctuation, and
tokenizes. Compare basic normalization, stopword removal, and lemmatization.
Negation words (`no`, `not`, `nor`) are retained when stopwords are removed.

We use basic WordNet lemmatization without POS tagging. The noun default maps
`books` to `book`, but leaves `running` unchanged. Both frozen corpora currently
select basic preprocessing. NLTK is still required for tokenization; downloaded
stopword/WordNet resources support the optional comparisons in this notebook.

These are Lab 1 preprocessing topics. We follow the topics without requiring
identical lab code.


In [5]:
sample = "How do I <b>NOT</b> lose my classes at https://portal.example.edu?"

print("raw               :", sample)
print("basic             :", preprocess_text(sample))
print("stopwords removed :", preprocess_text(sample, remove_stopwords=True))
print("lemmatized        :", preprocess_text(sample, lemmatize=True))

raw               : How do I <b>NOT</b> lose my classes at https://portal.example.edu?
basic             : how do i not lose my classes at
stopwords removed : not lose classes


lemmatized        : how do i not lose my class at


In [6]:
for name, options in PREPROCESSING_CONFIGS:
    print(f"{name:20} {options}")

basic                {'remove_stopwords': False, 'lemmatize': False}
stopwords_removed    {'remove_stopwords': True, 'lemmatize': False}
lemmatized           {'remove_stopwords': False, 'lemmatize': True}


## 4. Building the TF-IDF index

Only the **questions** are indexed. Answers never enter the similarity vectors.
`fit_transform()` learns the vocabulary and document frequencies from FAQ
questions and builds their matrix. A user query uses `transform()` with that
same fitted vocabulary and IDF; it must not refit the model or change its IDF.
Stored questions and queries use the same preprocessing options.

TF-IDF representation comes from **Lab 2**; cosine similarity comes from
**Lab 3**. Word2Vec and the sequence/Transformer models of later labs remain
optional extensions, outside this phase.


### A small TF-IDF and cosine calculation

The production vectorizer uses raw token counts (no sublinear TF), natural-log
smoothed IDF and L2 normalization. With $N$ FAQ questions and $DF(t)$ questions
containing term $t$:

$$TF(t,d)=\operatorname{count}(t,d), \qquad
IDF(t)=\log\!\left(\frac{1+N}{1+DF(t)}\right)+1$$

$$w_{t,d}=TF(t,d)\,IDF(t), \qquad
\widehat{\mathbf w}_d=\frac{\mathbf w_d}{\sqrt{\sum_t w_{t,d}^2}}$$

$$\cos(\mathbf q,\mathbf d)=
\frac{\mathbf q\cdot\mathbf d}{\|\mathbf q\|_2\|\mathbf d\|_2}$$

For two toy questions, `reset password password` and `change password`, the term
`password` has $DF=2$ and $IDF=1$; `reset` and `change` each have $DF=1$ and
$IDF=\log(3/2)+1\approx1.4055$. The repeated `password` has raw TF 2 in the first
question. The code below calculates every weight and checks it against
`build_tfidf_index`. For normalized nonzero vectors, cosine equals the dot product.
A zero query vector has no usable match and is rejected by the retrieval engine.


In [7]:
from sklearn.metrics.pairwise import cosine_similarity

toy_questions = ["reset password password", "change password"]
toy_faq = pd.DataFrame({"question": toy_questions})
toy_vectorizer, toy_matrix = build_tfidf_index(toy_faq)
terms = toy_vectorizer.get_feature_names_out()
counts = np.array([[text.split().count(term) for term in terms]
                   for text in toy_questions], dtype=float)
document_frequency = (counts > 0).sum(axis=0)
idf = np.log((1 + len(toy_questions)) / (1 + document_frequency)) + 1
weights = counts * idf
normalized = weights / np.linalg.norm(weights, axis=1, keepdims=True)
print("Raw term counts:")
print(pd.DataFrame(counts, columns=terms).to_string(index=False))
print("DF and smoothed IDF:")
print(pd.DataFrame({"term": terms, "DF": document_frequency, "IDF": idf}).to_string(index=False))
print("TF * IDF, before normalization:")
print(pd.DataFrame(weights, columns=terms).round(4).to_string(index=False))
print("L2-normalized FAQ vectors:")
print(pd.DataFrame(normalized, columns=terms).round(4).to_string(index=False))
assert np.allclose(idf, toy_vectorizer.idf_)
assert np.allclose(normalized, toy_matrix.toarray())

toy_query = "reset password"
query_counts = np.array([toy_query.split().count(term) for term in terms])
query_weights = query_counts * idf  # Reuse corpus IDF; do not fit on the query.
query_normalized = query_weights / np.linalg.norm(query_weights)
manual_cosines = normalized @ query_normalized
idf_before = toy_vectorizer.idf_.copy()
query_vector = toy_vectorizer.transform([toy_query])
library_cosines = cosine_similarity(query_vector, toy_matrix).ravel()
print("Normalized query:", np.round(query_normalized, 4))
print("Manual cosine scores:", np.round(manual_cosines, 4))
assert np.allclose(query_normalized, query_vector.toarray()[0])
assert np.allclose(manual_cosines, library_cosines)
assert np.array_equal(idf_before, toy_vectorizer.idf_)
print("Manual weights and cosine scores match the production vectorizer.")


Raw term counts:
 change  password  reset
    0.0       2.0    1.0
    1.0       1.0    0.0
DF and smoothed IDF:
    term  DF      IDF
  change   1 1.405465
password   2 1.000000
   reset   1 1.405465
TF * IDF, before normalization:
 change  password  reset
 0.0000       2.0 1.4055
 1.4055       1.0 0.0000
L2-normalized FAQ vectors:
 change  password  reset
 0.0000    0.8182  0.575
 0.8148    0.5797  0.000
Normalized query: [0.     0.5797 0.8148]
Manual cosine scores: [0.9428 0.3361]
Manual weights and cosine scores match the production vectorizer.


In [8]:
for name, bundle in datasets.items():
    config = bundle["config"]
    options = {
        "remove_stopwords": bool(config["remove_stopwords"]),
        "lemmatize": bool(config["lemmatize"]),
    }
    vectorizer, matrix = build_tfidf_index(bundle["faq"], options)
    bundle["options"] = options
    bundle["vectorizer"] = vectorizer
    bundle["matrix"] = matrix
    print(f"{config['display_name']:18} matrix {matrix.shape} "
          f"vocabulary {len(vectorizer.vocabulary_)}")

E-commerce FAQ     matrix (500, 680) vocabulary 680


University FAQ     matrix (500, 859) vocabulary 859


In [9]:
# Check vocabulary against questions processed exactly like the fitted index.
uni = datasets["university"]
vocab = set(uni["vectorizer"].vocabulary_)
question_words = set()
for question in uni["faq"]["question"]:
    question_words.update(preprocess_text(question, **uni["options"]).split())
print("vocabulary size           :", len(vocab))
print("vocabulary minus questions:", len(vocab - question_words))
assert vocab == question_words


vocabulary size           :

 859
vocabulary minus questions: 0


## 5. Retrieving an answer

`retrieve_tfidf` returns the top-k FAQs ranked by cosine similarity.
`answer_query` adds the threshold decision on top of that ranking.

In [10]:
def ask(corpus_name, question, top_k=3):
    """Run one query against a corpus and print the outcome."""
    bundle = datasets[corpus_name]
    result = answer_query(
        question,
        bundle["faq"],
        bundle["vectorizer"],
        bundle["matrix"],
        threshold=float(bundle["config"]["similarity_threshold"]),
        top_k=top_k,
        preprocessing_options=bundle["options"],
    )
    print(f"Query    : {question}")
    print(f"Corpus   : {bundle['config']['display_name']} "
          f"(threshold {result['threshold']:.2f})")
    if result["found"]:
        best = result["best_match"]
        print(f"ACCEPTED : similarity {best['similarity']:.4f}")
        print(f"Matched  : {best['question']}")
        print(f"Answer   : {best['answer'][:300]}")
    else:
        top = result["top_matches"]
        score = top[0]["similarity"] if top else 0.0
        print(f"REJECTED : best similarity {score:.4f}; below threshold or no vocabulary features")
    print("\nTop matches:")
    for rank, match in enumerate(result["top_matches"], start=1):
        print(f"  {rank}. {match['similarity']:.4f}  {match['question'][:80]}")
    print("-" * 78)
    return result


In [11]:
# An exact question is a sanity check, not a paraphrase evaluation.
faq = datasets["ecommerce"]["faq"].iloc[0]
result = ask("ecommerce", str(faq["question"]))
assert result["found"] and result["best_match"]["faq_id"] == int(faq["id"])
assert np.isclose(result["best_match"]["similarity"], 1.0)


Query    : How can I use my mobile number to login on the Flipkart mobile app?
Corpus   : E-commerce FAQ (threshold 0.58)
ACCEPTED : similarity 1.0000
Matched  : How can I use my mobile number to login on the Flipkart mobile app?
Answer   : To log into the Flipkart mobile app, update your mobile number from your Flipkart account on our website with these simples steps: 1. Go to My Account > Settings > Update Email/Mobile 2. Add your mobile number 3. Enter the OTP received on your registered mobile number 4. Select 'Save Changes'

Top matches:
  1. 1.0000  How can I use my mobile number to login on the Flipkart mobile app?
  2. 0.4849  Can I use my saved cards for making a payment on Flipkart's mobile site/app?
  3. 0.3865  Is 'Credit Card No Cost EMI' payment mode available on Flipkart's website and mo
------------------------------------------------------------------------------


In [12]:
# Verified university paraphrase: the expected FAQ asks how alerts arrive.
expected_id = 288
print("Expected FAQ:", datasets["university"]["faq"].set_index("id").loc[expected_id, "question"])
result = ask("university", "How do I receive university alerts?")
assert result["found"] and result["best_match"]["faq_id"] == expected_id


Expected FAQ: How will I receive alerts?
Query    : How do I receive university alerts?
Corpus   : University FAQ (threshold 0.46)
ACCEPTED : similarity 0.8010
Matched  : How will I receive alerts?
Answer   : All University of Toronto students, staff, faculty and librarians are automatically subscribed to receive UTAlert messages to their University of Toronto email addresses. UTAlert messages are also sent by SMS text message and mobile app push notifications and telephone for members who provide a phon

Top matches:
  1. 0.8010  How will I receive alerts?
  2. 0.6401  How often will I receive alerts?
  3. 0.2967  What number will alerts come from?
------------------------------------------------------------------------------


In [13]:
# Failure example: a natural returns/refund query is rejected and ranks a
# warranty question first. Do not present this as successful paraphrase matching.
result = ask("ecommerce", "How do I send a product back and get my money returned?")
assert not result["found"]
print("Observed limitation: the top candidate is FAQ", result["top_matches"][0]["faq_id"])


Query    : How do I send a product back and get my money returned?
Corpus   : E-commerce FAQ (threshold 0.58)
REJECTED : best similarity 0.3346; below threshold or no vocabulary features

Top matches:
  1. 0.3346  I didn't get a warranty card with my product. How can I get the warranty?
  2. 0.3240  What is a refurbished product and how do I identify it on Flipkart?
  3. 0.2536  How can I claim warranty for my product?
------------------------------------------------------------------------------
Observed limitation: the top candidate is FAQ 42


In [14]:
# Out-of-domain query, correctly rejected by the e-commerce corpus.
result = ask("ecommerce", "How do I train a puppy to sit?")
assert not result["found"]


Query    : How do I train a puppy to sit?
Corpus   : E-commerce FAQ (threshold 0.58)
REJECTED : best similarity 0.4614; below threshold or no vocabulary features

Top matches:
  1. 0.4614  How do I make a payment using a saved card?
  2. 0.4393  Do I need to have a driving license to drive a Bounce 2-wheeler?
  3. 0.4303  Do I need to have a driving license to ride a BGauss 2-wheeler?
------------------------------------------------------------------------------


The same puppy question is wrongly **accepted** by the university corpus.
With basic preprocessing it scores approximately **0.470**, exceeding its 0.46
threshold. FAQ 241 asks whether a teacher qualification is needed to apply to
OISE; the shared common words `do`, `i`, `a`, and `to` produce a lexical match.
This is a false acceptance, not a meaningful answer to the user's question.


In [15]:
result = ask("university", "How do I train a puppy to sit?")
assert result["found"] and result["best_match"]["faq_id"] == 241


Query    : How do I train a puppy to sit?
Corpus   : University FAQ (threshold 0.46)
ACCEPTED : similarity 0.4698
Matched  : Do I have to be a teacher to apply to OISE?
Answer   : Most programs don't require professional teacher certification. For admission to an MEd degree, generally, a year of professional education for teaching (or the equivalent in pedagogical content) and at least one year of relevant successful professional experience is helpful.

Top matches:
  1. 0.4698  Do I have to be a teacher to apply to OISE?
  2. 0.4617  How am I matched to a practicum?
  3. 0.4614  I have submitted my application but want to make a change, how do I do that?
------------------------------------------------------------------------------


### Why the threshold and vocabulary check matter

A query with vocabulary overlap can produce an irrelevant highest-ranked FAQ.
The threshold can reject that weak match. An all-OOV query has a zero vector
and **always** produces no matches, even at threshold 0.00. The batch evaluator
also masks it out of Top-1/Top-3 accuracy to avoid credit from zero-score ties.


In [16]:
bundle = datasets["university"]
for query in ["What is the current price of Bitcoin?", "zzqqxx wwvvuu ttrrpp"]:
    print("Query:", query)
    for threshold in [0.0, float(bundle["config"]["similarity_threshold"])]:
        result = answer_query(
            query, bundle["faq"], bundle["vectorizer"], bundle["matrix"],
            threshold=threshold, top_k=1,
            preprocessing_options=bundle["options"],
        )
        verdict = "ACCEPTED" if result["found"] else "REJECTED"
        score = result["top_matches"][0]["similarity"] if result["top_matches"] else 0.0
        print(f"threshold {threshold:.2f} -> {verdict} (best similarity {score:.4f})")
        if query.startswith("zzqqxx"):
            assert not result["found"] and result["top_matches"] == []


Query: What is the current price of Bitcoin?


threshold 0.00 -> ACCEPTED (best similarity 0.3643)
threshold 0.46 -> REJECTED (best similarity 0.3643)
Query: zzqqxx wwvvuu ttrrpp
threshold 0.00 -> REJECTED (best similarity 0.0000)
threshold 0.46 -> REJECTED (best similarity 0.0000)


## 6. Choosing preprocessing and threshold

The selection functions receive **validation data only**.

- **Step A:** `select_preprocessing` compares answerable validation Top-1
  accuracy, then Top-3, then the simpler preprocessing configuration.
- **Step B:** `tune_threshold` orchestrates Step A, then sweeps 0.00 to 1.00 in
  0.01 steps only for the selected configuration. Maximize the mean of
  answerable acceptance and unanswerable rejection; ties prefer the higher
  threshold. This stage measures answerability decisions, not answer correctness.

The configuration files were frozen by `python evaluate.py tune`. The cell below
reproduces the selection in memory and checks the saved choices without writing
files. It does not load the synthetic test queries.


In [17]:
tuning = {}
for name, bundle in datasets.items():
    validation = load_query_dataset(
        bundle["dir"] / "validation_queries.csv", set(bundle["faq"]["id"])
    )
    result = tune_threshold(validation, bundle["faq"])
    tuning[name] = result
    print(f"=== {bundle['config']['display_name']} ({len(validation)} validation queries) ===")
    print("Step A: answerable retrieval")
    print(pd.DataFrame(result["preprocessing_comparison"])[
        ["config", "top1_accuracy", "top3_accuracy"]
    ].to_string(index=False))
    row = result["per_config_best"][0]
    print(f"Step B: {row['config']} at {row['threshold']:.2f}, "
          f"balanced acceptance/rejection {row['score']:.4f}")
    assert result["selected_config"] == bundle["config"]["preprocessing_config"]
    assert result["selected_options"] == bundle["options"]
    assert result["selected_threshold"] == bundle["config"]["similarity_threshold"]


=== E-commerce FAQ (50 validation queries) ===
Step A: answerable retrieval
           config  top1_accuracy  top3_accuracy
            basic            0.9            0.9
stopwords_removed            0.9            0.9
       lemmatized            0.9            0.9
Step B: basic at 0.58, balanced acceptance/rejection 0.8500


=== University FAQ (50 validation queries) ===
Step A: answerable retrieval
           config  top1_accuracy  top3_accuracy
            basic       0.900000            1.0
stopwords_removed       0.866667            1.0
       lemmatized       0.900000            1.0
Step B: basic at 0.46, balanced acceptance/rejection 0.9167


Both domains select **basic** preprocessing. On university validation queries,
basic and lemmatized tie at 0.900 Top-1 and 1.000 Top-3; basic wins the simplicity
tie. Stopword removal reaches 0.867 Top-1. E-commerce ties across all three
configurations at 0.900 Top-1 and Top-3. These measurements do not support the
old claim that lemmatization improves university retrieval.


In [18]:
# How the score varies with the threshold, for the selected configuration.
for name, result in tuning.items():
    sweep = pd.DataFrame(result["sweep"])
    best = sweep[sweep["config"] == result["selected_config"]]
    peak = best[best["score"] == best["score"].max()]
    print(f"{name}: best score {best['score'].max():.4f} is reached at "
          f"thresholds {peak['threshold'].min():.2f}-{peak['threshold'].max():.2f}; "
          f"the highest is chosen ({result['selected_threshold']:.2f})")

ecommerce: best score 0.8500 is reached at thresholds 0.50-0.58; the highest is chosen (0.58)
university: best score 0.9167 is reached at thresholds 0.46-0.46; the highest is chosen (0.46)


## 7. Synthetic benchmark with frozen settings

Apply the saved configuration without retuning on these queries. The synthetic
test sets have already been inspected during development and review: these are
reproducible benchmark results, not an untouched, one-time final assessment.
The computed metrics below are checked against the regenerated JSON reports.


In [19]:
reports = {}
for name, bundle in datasets.items():
    test = load_query_dataset(
        bundle["dir"] / "test_queries.csv", set(bundle["faq"]["id"])
    )
    report = evaluate_tfidf(
        test, bundle["faq"], bundle["vectorizer"], bundle["matrix"],
        threshold=float(bundle["config"]["similarity_threshold"]),
        preprocessing_options=bundle["options"],
    )
    saved = json.loads((ROOT / "reports" / f"{name}_evaluation.json").read_text(encoding="utf-8"))
    for key, value in report.items():
        assert saved[key] == value, (name, key)
    reports[name] = report

summary = pd.DataFrame({
    datasets[name]["config"]["display_name"]: {
        "Preprocessing": datasets[name]["config"]["preprocessing_config"],
        "Threshold": datasets[name]["config"]["similarity_threshold"],
        "Correct answer rate": round(r["correct_answer_rate"], 3),
        "Accepted but wrong": r["accepted_wrong_count"],
        "Top-1 accuracy": round(r["top1_accuracy"], 3),
        "Top-3 accuracy": round(r["top3_accuracy"], 3),
        "Mean similarity of correct Top-1": round(r["mean_similarity_correct_top1"], 4),
        "Answerable acceptance rate": round(r["answerable_acceptance_rate"], 3),
        "Unanswerable rejection rate": round(r["unanswerable_rejection_rate"], 3),
        "False acceptances": r["false_acceptance_count"],
        "False rejections": r["false_rejection_count"],
    }
    for name, r in reports.items()
})
summary


,E-commerce FAQ,University FAQ
Preprocessing,basic,basic
Threshold,0.58,0.46
Correct answer rate,0.82,0.833
Accepted but wrong,1,17
Top-1 accuracy,0.953,0.86
Top-3 accuracy,0.98,0.993
Mean similarity of correct Top-1,0.74,0.7602
Answerable acceptance rate,0.827,0.947
Unanswerable rejection rate,0.96,0.84
False acceptances,2,8


### Reading these numbers

- **Top-1/Top-3** measure retrieval before the threshold, on answerable queries
  with vocabulary features. Zero vectors receive no retrieval credit.
- **Correct answer rate** = number of answerable queries with correct Top-1
  **and** acceptance, divided by all answerable queries. It is 0.833 for
  university and 0.820 for e-commerce.
- **Accepted but wrong** counts answerable queries accepted with a different
  FAQ: 17 university and 1 e-commerce. It is separate from false acceptance,
  which counts unanswerable queries that receive any answer.
- University Top-3 is 0.993 versus Top-1 0.860: ranking similar FAQ intents
  remains a limitation. E-commerce rejects 26 answerable queries at 0.58.
  Acceptance alone and Top-1 alone therefore do not describe answer delivery.


## 8. Error analysis

These are the cases the teacher is most likely to ask about, so they are shown
rather than summarised away.

In [20]:
for name, report in reports.items():
    title = datasets[name]["config"]["display_name"]
    print(f"=== {title}: incorrect Top-1 retrievals ===")
    for example in report["incorrect_retrieval_examples"][:3]:
        print(f"  Query     : {example['query'][:95]}")
        print(f"  Expected  : FAQ {example['expected_faq_id']}")
        print(f"  Retrieved : FAQ {example['retrieved_faq_id']} - "
              f"{example['retrieved_question'][:70]}")
        print(f"  Similarity: {example['similarity']:.4f}  "
              f"accepted={example['accepted']}\n")

=== E-commerce FAQ: incorrect Top-1 retrievals ===
  Query     : What should be understood about the guarantee being provided by Ather?
  Expected  : FAQ 330
  Retrieved : FAQ 341 - What after sales services are being provided by Ather for 2-wheelers?
  Similarity: 0.4680  accepted=False

  Query     : What should be understood about Money on Shipment?
  Expected  : FAQ 294
  Retrieved : FAQ 324 - I have a query about the EMI charge. What should I do?
  Similarity: 0.3618  accepted=False

  Query     : What steps should someone follow to settle the amount for my purchase?
  Expected  : FAQ 302
  Retrieved : FAQ 205 - Is there a minimum purchase amount for Flipkart Quick orders?
  Similarity: 0.3863  accepted=False

=== University FAQ: incorrect Top-1 retrievals ===
  Query     : What investigations are carried out in Germanic Literature, Culture and Theory?
  Expected  : FAQ 47
  Retrieved : FAQ 97 - what subjects will I study in Germanic Literature, Culture and Theory 
  Similarity: 0

In [21]:
# Where do answerable and unanswerable queries actually sit on the score scale?
for name, bundle in datasets.items():
    test = load_query_dataset(
        bundle["dir"] / "test_queries.csv", set(bundle["faq"]["id"])
    )
    scores = []
    for query in test["query"]:
        matches = retrieve_tfidf(
            query, bundle["faq"], bundle["vectorizer"], bundle["matrix"],
            top_k=1, preprocessing_options=bundle["options"],
        )
        scores.append(matches[0]["similarity"] if matches else 0.0)
    test = test.assign(top_score=scores)
    answerable = test[test["is_answerable"]]["top_score"]
    unanswerable = test[~test["is_answerable"]]["top_score"]
    print(f"{bundle['config']['display_name']} "
          f"(threshold {bundle['config']['similarity_threshold']:.2f})")
    print(f"  answerable   mean {answerable.mean():.3f}  "
          f"min {answerable.min():.3f}  max {answerable.max():.3f}")
    print(f"  unanswerable mean {unanswerable.mean():.3f}  "
          f"min {unanswerable.min():.3f}  max {unanswerable.max():.3f}")
    print(f"  the two ranges overlap, which is why no threshold scores 1.000\n")

E-commerce FAQ (threshold 0.58)
  answerable   mean 0.726  min 0.362  max 0.995
  unanswerable mean 0.387  min 0.232  max 0.641
  the two ranges overlap, which is why no threshold scores 1.000



University FAQ (threshold 0.46)
  answerable   mean 0.737  min 0.299  max 0.984
  unanswerable mean 0.369  min 0.222  max 0.525
  the two ranges overlap, which is why no threshold scores 1.000



The score overlap explains why a single cosine threshold cannot perfectly
separate answerable from unanswerable queries on this benchmark. A short
out-of-domain query can share common words with an FAQ. Word2Vec or Transformer
representations are possible later comparisons, not guaranteed fixes.

The validators check schemas, counts, labels, and duplicate rules; they cannot
certify that every source answer or generated paraphrase is semantically correct.


## 9. Ask your own question

Change the corpus and the question below and re-run the cell.

In [22]:
ask("university", "What are the entry criteria for a graduate programme?")
ask("ecommerce", "My package has not arrived yet, what should I do?")
None

Query    : What are the entry criteria for a graduate programme?
Corpus   : University FAQ (threshold 0.46)
REJECTED : best similarity 0.3510; below threshold or no vocabulary features

Top matches:
  1. 0.3510  what are the admission requirements for Counselling Psychology at UOFT?
  2. 0.3499  what are the admission requirements for Spanish at UOFT?
  3. 0.3327  What are the admission requirements?
------------------------------------------------------------------------------
Query    : My package has not arrived yet, what should I do?
Corpus   : E-commerce FAQ (threshold 0.58)
REJECTED : best similarity 0.5397; below threshold or no vocabulary features

Top matches:
  1. 0.5397  What should I do if my order is approved but hasn't been shipped yet?
  2. 0.4477  My package shows as delivered but I can't find it, what should I do?
  3. 0.3795  What should I do if I find the package open or tampered on delivery?
---------------------------------------------------------------------------

## 10. Summary and human evaluation

The summary table in section 7 is computed from the frozen configurations and
checked against saved reports. Both domains use basic preprocessing; thresholds
remain 0.46 (university) and 0.58 (e-commerce).

This project combines Lab 1 preprocessing, Lab 2 TF-IDF and Lab 3 cosine
similarity through a reusable function-based engine. It retrieves stored answers
and runs offline after setup. The regression suite covers OOV handling,
selection, metrics, and the manual-evaluation command.

**Human evaluation is pending.** Header-only CSVs and a collection guide are in
`data/manual_evaluation/`. The target is 20 answerable + 10 unanswerable questions
per domain. Write from intent descriptions without looking at FAQ wording;
independently check labels and record authorship. Keep illustrative examples out
of measured files. Run `python evaluate.py manual` with the frozen settings.
Empty files produce no performance figures; populated files use separate reports.

The existing benchmark uses synthetic paraphrases. These are distinct from the
source questions but sometimes awkward or semantically imperfect. No claim of
human-written performance is made until actual team-written queries exist.
